<a href="https://colab.research.google.com/github/edwardsnj/pride-study-retrieval-cofest-2026/blob/main/notebooks/BaseModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# files...
BASEURL = "https://edwardslab.bmcb.georgetown.edu/~nedwards/dropbox/6ItUS2tEdC/"
csvfile = "pride-embeddings-openai-3-small.csv"
fthfile = "pride-embeddings-openai-3-small.fth"

GITHUB = "https://raw.githubusercontent.com/EdwardsLabProjects/pride-study-retrieval-cofest-2026/refs/heads/main/data/"
trueposfile = "truepos.txt"
truenegfile = "trueneg.txt"

# download...
for f in [csvfile, fthfile]:
  !wget -nc {BASEURL+f}
for f in [trueposfile, truenegfile]:
  !wget -nc {GITHUB+f}


--2026-07-17 17:24:18--  https://edwardslab.bmcb.georgetown.edu/~nedwards/dropbox/6ItUS2tEdC/pride-embeddings-openai-3-small.csv
Resolving edwardslab.bmcb.georgetown.edu (edwardslab.bmcb.georgetown.edu)... 141.161.14.163
Connecting to edwardslab.bmcb.georgetown.edu (edwardslab.bmcb.georgetown.edu)|141.161.14.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 161384101 (154M) [text/csv]
Saving to: ‘pride-embeddings-openai-3-small.csv’

pride-embeddings-op 100%[===================>] 153.91M  47.4MB/s    in 3.6s    

2026-07-17 17:24:22 (42.3 MB/s) - ‘pride-embeddings-openai-3-small.csv’ saved [161384101/161384101]

--2026-07-17 17:24:22--  https://edwardslab.bmcb.georgetown.edu/~nedwards/dropbox/6ItUS2tEdC/pride-embeddings-openai-3-small.fth
Resolving edwardslab.bmcb.georgetown.edu (edwardslab.bmcb.georgetown.edu)... 141.161.14.163
Connecting to edwardslab.bmcb.georgetown.edu (edwardslab.bmcb.georgetown.edu)|141.161.14.163|:443... connected.
HTTP request sent, 

In [5]:
import pandas as pd

emb = pd.read_feather(fthfile)
md = pd.read_csv(csvfile)

tp = set(open(trueposfile).read().split())
tn = set(open(truenegfile).read().split())

# Keep only the ones that there are embeddings for
tp &= set(emb.columns)
tn &= set(emb.columns)

assert len(set(tp) & set(tn)) == 0, "TP and TN should not intersect!"

# Print out some details...
print(f"Embeddings: {emb.shape}")
print(f"True Pos: {len(tp)}")
print(f"True Neg: {len(tn)}")

Embeddings: (1536, 36696)
True Pos: 84
True Neg: 47


In [8]:
import random

import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils import shuffle

def train_document_classifier(embeddings, seeds, neg_seeds, test=0.2, bgsize=1.0, random_state=None, **kwargs):
    seeds = list(set(seeds)&set(embeddings.columns))
    neg_seeds = list(set(neg_seeds)&set(embeddings.columns))
    bg = list(set(embeddings.columns)-set(seeds))
    nbgsel = int(round(len(seeds)*bgsize))
    if len(bg) < len(neg_seeds):
        raise ValueError("Not enough background embeddings to create selected background.")
    if len(bg) < nbgsel:
        raise ValueError("Not enough background embeddings to create selected background.")

    if test > 0.0:
        # 1. Split the original seed set
        pos_train_accs, pos_test_accs = train_test_split(
            seeds,
            test_size=test,
            random_state=random_state
        )
        have_test = True
    else:
        pos_train_accs = list(seeds)
        pos_test_accs = []
        have_test = False

    selected_accessions = list(seeds)
    train_accessions = list(pos_train_accs)
    test_accessions = list(pos_test_accs)

    num_train_samples = len(pos_train_accs)
    num_bg_train_samples = int(round(num_train_samples*bgsize))
    num_test_samples = len(pos_test_accs)
    num_bg_test_samples = nbgsel-num_bg_train_samples

    random.seed(random_state)

    selbg = neg_seeds + list(random.sample(list(set(bg)-set(neg_seeds)),
                                           nbgsel-len(neg_seeds)))
    random.shuffle(selbg)
    seltrainbg = selbg[:num_bg_train_samples]
    seltestbg = selbg[num_bg_train_samples:]

    selected_accessions += selbg
    train_accessions += seltrainbg
    test_accessions += seltestbg

    np.random.seed(random_state)

    pos_train = embeddings[pos_train_accs].values.T
    if have_test:
        pos_test = embeddings[pos_test_accs].values.T

    # Extract the exact number of negative examples needed for testing and training

    neg_train =  embeddings[seltrainbg].values.T
    if have_test:
        neg_test = embeddings[seltestbg].values.T

    # 3. Assemble the training and testing datasets
    # Combine positive (1) and negative (0) features
    X_train = np.vstack([pos_train, neg_train])
    y_train = np.concatenate([np.ones(num_train_samples), np.zeros(num_bg_train_samples)])

    if have_test:
        X_test = np.vstack([pos_test, neg_test])
        y_test = np.concatenate([np.ones(num_test_samples), np.zeros(num_bg_test_samples)])

    # 4. Shuffle the datasets so labels are randomized (not all 1s followed by all 0s)
    X_train, y_train = shuffle(X_train, y_train, random_state=random_state)
    if have_test:
        X_test, y_test = shuffle(X_test, y_test, random_state=random_state)

    print(f"Training data shape: {X_train.shape} (Positives: {num_train_samples}, Negatives: {num_bg_train_samples})")
    if have_test:
        print(f"Testing data shape: {X_test.shape} (Positives: {num_test_samples}, Negatives: {num_bg_test_samples})")

    # 5. Initialize and train the Logistic Regression model
    model = LogisticRegression(max_iter=1000, random_state=random_state, **kwargs)
    model.fit(X_train, y_train)

    # 6. Evaluate the model on the withheld test set
    if not have_test:
        return model, train_accessions, test_accessions

    y_pred = model.predict(X_test)

    print("\n--- Model Evaluation ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("\nClassification Report:")
    # target_names let us interpret the 1s and 0s easily in the terminal
    print(classification_report(y_test, y_pred, target_names=["Background (0)", "Seed-like (1)"]))

    return model, train_accessions, test_accessions

In [12]:
extra_args = dict(test=0.2, C=10.0, penalty='l1', solver='liblinear', bgsize=25)
trained_model, train_acc, test_acc = train_document_classifier(emb, tp, tn, random_state=None, **extra_args)
print("NZ Coeffs:",sum(*[1*(xi!=0) for xi in trained_model.coef_]))

Training data shape: (1742, 1536) (Positives: 67, Negatives: 1675)
Testing data shape: (442, 1536) (Positives: 17, Negatives: 425)

--- Model Evaluation ---
Accuracy: 0.9706

Classification Report:
                precision    recall  f1-score   support

Background (0)       0.97      1.00      0.98       425
 Seed-like (1)       0.75      0.35      0.48        17

      accuracy                           0.97       442
     macro avg       0.86      0.67      0.73       442
  weighted avg       0.97      0.97      0.97       442

NZ Coeffs: 75
